#### PDFの読込とカスタマイズ
基本的なベクトルDBを使用したRAGのカスタマイズ

LlamaIndexの主要なカスタマイズ項目で、
日本語のPDFファイルを読んでLLMアプリで扱うのに役立つ

◆準備

In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")
os.environ['OPENAI_API_KEY']  = os.environ['API_KEY']
# API_KEYという環境変数の値をOPENAI_API_KEYにコピー

# モデル名
MODEL_NAME = "gpt-4o-mini"


◆インデックスの構築

PDFドキュメントの読込んで、インデックスをつくる際に
日本語に適した形へカスタマイズ

In [2]:
# PDFドキュメントの読込にも、
# SimpleDirectoryReader が使える

documents = SimpleDirectoryReader('./data/pdf').load_data()


NodeParser の設定で日本語の特徴にあわせてRAGの性能を上げる　↓

In [ ]:
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core import Settings
import tiktoken

# NodeParserの作成
node_parser = SentenceSplitter(
    separator="。",         # 区切り文字を「。」に
    chunk_size=256,     # チャンクサイズのデフォ1024を小さい値から調整
    chunk_overlap=16,   # デフォ20。前後のチャンクをオーバーラップ。文脈を読みやすく
    tokenizer=tiktoken.encoding_for_model(MODEL_NAME).encode)
# tokenizer：チャンクを正しいトークン数で数えるために使用。
# 言語モデルに合わせてエンコーディングを指定します。
# tiktoken.encoding_for_model(MODEL_NAME) で言語モデルに合った
# トークナイザーが取得できます（gpt-4o-mini の場合は o200k_base ）
# OpenAIのモデルごとに、最適なトークナイザーがある。

# 言語モデルの指定
llm = OpenAI(model=MODEL_NAME, temperature=0.3)

# 設定に反映
Settings.llm = llm
Settings.node_parser = node_parser   # ここで設定を渡してる

# Indexの構築（ここでベクトル化・enbeddingがされてる）
index = VectorStoreIndex.from_documents(documents)


◆作ったインデックスの保存と読込

In [ ]:
# ストレージに保存
index.storage_context.persist("./storage01")

In [5]:
# ストレージからインデックスをロード（復元）
from llama_index.core import StorageContext, load_index_from_storage

# ストレージコンテキストの作成
storage_context = StorageContext.from_defaults(
    persist_dir="./storage01"
    )

# Indexのロード
index = load_index_from_storage(storage_context)


◆プロンプトテンプレートの日本語化

LlamaIndexに組み込まれているプロンプトは英語。うっかり英語で回答されないように、日本語のプロンプトを組み込んでおく

In [7]:
sys_prompt_str = """
事前知識ではなく、常に提供されたコンテキスト情報を使用して質問に回答してください。
回答内でコンテキストを直接参照しないでください。
「コンテキストに基づいて」や「コンテキスト情報は」、またはそれに類するような記述は避けてください。
"""

qa_prompt_str = """
コンテキスト情報は以下の通りです。
---------------------
{context_str}
---------------------
事前知識ではなくコンテキスト情報を使用して、質問に回答してください。
質問: {query_str}
回答："""

refine_prompt_str = """
元の回答を (必要な場合のみ) 以下のコンテキストで改良する機会があります。
-----------
{context_msg}
-----------
新しいコンテキストが与えられた場合、元の回答を改良して、質問 {query_str} に適切に回答します。
コンテキストが役に立たない場合は、元の回答を再度出力します。
元の回答: {existing_answer}"""

In [ ]:
# 日本語化されたプロンプトのベースを使って、
# プロンプトテンプレートの作成


from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core import ChatPromptTemplate

# テキストQAテンプレートの作成
chat_text_qa_msgs = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=sys_prompt_str),
    ChatMessage(
        role=MessageRole.USER,
        content=qa_prompt_str),
]
text_qa_template = ChatPromptTemplate(chat_text_qa_msgs)

# リファインテンプレートの作成
chat_refine_msgs = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=sys_prompt_str),
    ChatMessage(
        role=MessageRole.USER,
        content=refine_prompt_str),
]
refine_template = ChatPromptTemplate(chat_refine_msgs)

◆チャットエンジンの作成

In [ ]:
# 作った日本穂テンプレートを使った、Chat Engineの作成
chat_engine = index.as_chat_engine(
    chat_mode="openai",
    llm=llm,
    similarity_top_k=3,   # 関連性の高いノードを何個出すか？
    text_qa_template=text_qa_template,   # パラメータに名前をそろえてる
    refine_template=refine_template,
)

◆テスト1回目

In [13]:
#テスト1回目

# 質問：1回目
response = chat_engine.stream_chat("公共交通機関の交通費の上限は？")

"""
for token in response.response_gen:
    print(token, end="")

「response_gen メソッドは、チャットエンジンにおける streaming=True と
矛盾しているため、今後変更される可能性があります。」
将来的には：
（1）response 自体がイテレート可能になる？
（2） response.stream_tokens() みたいな別メソッドが登場する？
かもしれない。公式の Usage Pattern をチェック。

"""

for token in response.response_gen:
    print(token, end="", flush=True)

公共交通機関の交通費の上限は、月額3万円まで支給されます。

◆引用元を表示して確認

In [14]:
# 1回目の引用元を表示
for source in response.sources:
    for source_node in source.raw_output.source_nodes:
        print("ファイル名：", source_node.metadata["file_name"])
        print("関連度スコア:", source_node.score)
        print("テキスト：")
        print(source_node.node.text)
        print("-" * 50)  # 区切り線

ファイル名： 02賃金規則.pdf
関連度スコア: 0.873949299033571
テキスト：
通勤⼿当
通勤にかかる交通費は、実際の経路に基づき⽀給されます。
公共交通機関の利⽤の場合は、最安経路をもとに⽉額上限 3 万円まで⽀給します。
⾃家⽤⾞での通勤が必要な場合は、事前に⼈事部へ申請してください。駐⾞場の使
⽤料やガソリン代の⼀部が⽀給される場合もあります。
2.
--------------------------------------------------
ファイル名： 02賃金規則.pdf
関連度スコア: 0.8153141230470965
テキスト：
住宅⼿当
会社から通勤に 1 時間以上かかる場合、住宅⼿当として⽉額 1 万円が⽀給されます。
住宅⼿当を受けるためには、賃貸契約書など、居住地を証明できる書類の提出が必
要です。
3. 家族⼿当
扶養家族がいる従業員には、家族⼿当が⽀給されます。
配偶者には⽉額 5,000 円、⼦供⼀⼈につき⽉額 3,000 円が⽀給されます（上限︓⼦供 3
⼈まで）。
4.
--------------------------------------------------
ファイル名： 02賃金規則.pdf
関連度スコア: 0.7867391734447402
テキスト：
時間外⼿当（残業代）
所定の勤務時間を超えて働いた場合は、残業⼿当が⽀給されます。
残業⼿当の割増率は、法令に基づき計算されます。通常の時間外労働は 1.25 倍、深
夜時間帯（午後 10 時以降）の残業は 1.5 倍となります。
5. 休⽇出勤⼿当
--------------------------------------------------


◆チャットエンジンに問い合わせ　2回目

In [15]:
# 質問：2回目
response = chat_engine.stream_chat(
    "交通費以外の手当にはどのようなものがありますか？"
    )

for token in response.response_gen:
    print(token, end="", flush=True)


交通費以外の手当には、以下のものがあります。

1. **住宅手当**: 通勤に1時間以上かかる場合、月額1万円が支給されます。受給には居住地を証明できる書類の提出が必要です。

2. **家族手当**: 扶養家族がいる従業員には支給され、配偶者には月額5,000円、子供1人につき月額3,000円が支給されます（上限は子供3人まで）。

3. **時間外手当（残業代）**: 所定の勤務時間を超えて働いた場合に支給され、通常の時間外労働は1.25倍、深夜時間帯の残業は1.5倍となります。

4. **休日出勤手当**: 休日に出勤した場合に支給される手当です。

◆引用元を表示して確認2回目

In [16]:
# 引用元を表示
for source in response.sources:
    for source_node in source.raw_output.source_nodes:
        print("ファイル名：", source_node.metadata["file_name"])
        print("関連度スコア:", source_node.score)
        print("テキスト：")
        print(source_node.node.text)
        print("-" * 50)  # 区切り線

ファイル名： 02賃金規則.pdf
関連度スコア: 0.8594400253843295
テキスト：
通勤⼿当
通勤にかかる交通費は、実際の経路に基づき⽀給されます。
公共交通機関の利⽤の場合は、最安経路をもとに⽉額上限 3 万円まで⽀給します。
⾃家⽤⾞での通勤が必要な場合は、事前に⼈事部へ申請してください。駐⾞場の使
⽤料やガソリン代の⼀部が⽀給される場合もあります。
2.
--------------------------------------------------
ファイル名： 02賃金規則.pdf
関連度スコア: 0.8170737209675953
テキスト：
住宅⼿当
会社から通勤に 1 時間以上かかる場合、住宅⼿当として⽉額 1 万円が⽀給されます。
住宅⼿当を受けるためには、賃貸契約書など、居住地を証明できる書類の提出が必
要です。
3. 家族⼿当
扶養家族がいる従業員には、家族⼿当が⽀給されます。
配偶者には⽉額 5,000 円、⼦供⼀⼈につき⽉額 3,000 円が⽀給されます（上限︓⼦供 3
⼈まで）。
4.
--------------------------------------------------
ファイル名： 02賃金規則.pdf
関連度スコア: 0.8039145746655675
テキスト：
時間外⼿当（残業代）
所定の勤務時間を超えて働いた場合は、残業⼿当が⽀給されます。
残業⼿当の割増率は、法令に基づき計算されます。通常の時間外労働は 1.25 倍、深
夜時間帯（午後 10 時以降）の残業は 1.5 倍となります。
5. 休⽇出勤⼿当
--------------------------------------------------


ベクトルDBを用いたRAG

ベクトル検索型 RAG・・・今回学習したようなやつ。Pythonのメモリ上に保持される軽量な検索エンジンをllama_indexが利用してる

RAG実装でよく用いられる仕組みとしては、下記がある
ベクトル検索用ライブラリ、ベクトルDB（FAISS, Pinecone など）
　FAISS・・・軽量・組み込み向け。Metaが開発したローカルで使えるベクトル検索ライブラリ
　Pinecone・・商用向け。クラウド上で動く、完全管理型のベクトルデータベース

Hybrid RAG（ハイブリッド RAG）って　
（2015年ではこれが主流）

・ベクトル検索 + キーワード検索（Hybrid Search）
ベクトル検索と従来のキーワード検索を組み合わせる手法です。キーワード検索の精度とベクトル検索の柔軟性を補完し、精度向上が図れるため、広く用いられています。
ベクトル検索とキーワード検索を同時に行って、結果を統合するとか。

・マルチソース検索
複数のデータベースを組み合わせ（例：ベクトルデータベース + SQLデータベース）、各ソースからの情報を取得・統合して回答を生成する方法
↓それぞれを別の方法でとってくるイメージ
　社内ナレッジ → ベクトル検索
　勤怠データ → SQL
　天気情報 → 外部API

グラフRAG　
ノード（情報の単位）同士の関係性（リンク）をグラフ構造で表現できる賢いRAG.文脈や因果関係、階層構造を活かせる

llama_index でのグラフRAGもある
llama_index では、以下のような機能：

KnowledgeGraphIndex：エンティティ間の関係を抽出してグラフ化
ComposableGraph：複数のインデックスを階層的に構成

マルチモーダル RAG
検索材料が音声や図も可能。
例えば材料が音声なら「誰が何を言った？」に答えられるような